# Trustworthy ML for Scientific Assays: Sensitivity, Stability, and Explainability in Gene Expression Data

**Author:** Liz Dauster  
**Course:** DSE 510 Project  
**Due date:** May 8, 2026  
**Dataset:** GEO accession [GSE203539](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE203539)


## Executive Summary
This notebook extends prior DESeq2 analysis into a Python-first trustworthy-ML workflow focused on decision support in bioscience assay contexts.

## 1) Business / Scientific Decision Context (So What?)
Translate noisy high-dimensional assay outputs into robust, explainable signals for scientific prioritization and replication decisions.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.inspection import permutation_importance
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set(style='whitegrid', context='talk')

## 2) Data Source, Attribution, and Prompt Alignment
- GEO accession: GSE203539  
- Public data with citation and publication attribution  
- Continuation of your R/DESeq2 workflow (not a repeat)  
- Covers visualization, sampling, aggregation, cleaning, slicing, feature engineering, text matching, and sensitivity analysis

In [ ]:
# Configure paths
DATA_DIR = Path('./data')
COUNTS_PATH = DATA_DIR / 'selected_counts_transformed.tsv'   # TODO
META_PATH = DATA_DIR / 'sample_metadata.tsv'                 # TODO
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load inputs
counts = pd.read_csv(COUNTS_PATH, sep='\t', index_col=0)  # rows=Ensembl IDs, cols=samples
meta = pd.read_csv(META_PATH, sep='\t')
meta = meta.rename(columns={'sample':'sample_id','Sample':'sample_id','cell_type':'CellType','treatment':'Treatment'})
required_cols = {'sample_id','CellType','Treatment'}
assert required_cols.issubset(meta.columns), f'Missing columns: {required_cols - set(meta.columns)}'

# Align sample IDs
counts.columns = counts.columns.astype(str)
meta['sample_id'] = meta['sample_id'].astype(str)
shared = sorted(set(counts.columns).intersection(set(meta['sample_id'])))
counts = counts[shared]
meta = meta.set_index('sample_id').loc[shared].reset_index()
print('Counts shape:', counts.shape)
print('Metadata shape:', meta.shape)

## 3) Exploratory Structure (Continuation QA)

In [ ]:
sample_corr = np.corrcoef(counts.T)
plt.figure(figsize=(10,8))
sns.heatmap(sample_corr, cmap='vlag', center=0, xticklabels=counts.columns, yticklabels=counts.columns)
plt.title('Sample Correlation Heatmap (Transformed Counts)')
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.decomposition import PCA
pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(counts.T.values)
pca_df = meta.copy()
pca_df['PC1'] = pcs[:,0]
pca_df['PC2'] = pcs[:,1]
plt.figure(figsize=(10,7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='CellType', style='Treatment', s=140)
plt.title('PCA: Transformed Expression by Cell Type/Treatment')
plt.tight_layout(); plt.show()

## 4) Task Definition and Modeling

In [ ]:
def build_task_dataset(counts_df, meta_df, cell_type, positive_label, negative_label):
    sub = meta_df[(meta_df['CellType'] == cell_type) & (meta_df['Treatment'].isin([positive_label, negative_label]))].copy()
    y = (sub['Treatment'] == positive_label).astype(int).values
    X = counts_df[sub['sample_id'].tolist()].T
    return X, y, sub

TASKS = [
    ('AEC_WSN_vs_mock', 'AEC', 'WSN', 'mock'),
    ('AM_WSN_vs_mock', 'AM', 'WSN', 'mock')
]

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=25, random_state=RANDOM_STATE)
log_reg = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('var_filter', VarianceThreshold(0.0)),
    ('scale', StandardScaler()),
    ('model', LogisticRegression(penalty='l1', solver='saga', C=0.2, max_iter=5000, random_state=RANDOM_STATE))
])
rf = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('var_filter', VarianceThreshold(0.0)),
    ('model', RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])
scoring = {'roc_auc':'roc_auc','pr_auc':'average_precision','balanced_acc':'balanced_accuracy','f1':'f1'}
rows = []
for task_name, ct, pos, neg in TASKS:
    X, y, _ = build_task_dataset(counts, meta, ct, pos, neg)
    for model_name, model in [('LogReg_L1', log_reg), ('RandomForest', rf)]:
        out = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        row = {'task':task_name, 'model':model_name}
        for m in scoring:
            row[f'{m}_mean'] = out[f'test_{m}'].mean()
            row[f'{m}_std'] = out[f'test_{m}'].std()
        rows.append(row)
results_df = pd.DataFrame(rows).sort_values(['task','roc_auc_mean'], ascending=[True,False])
results_df

## 5) Sensitivity and Explainability

In [ ]:
best = results_df.sort_values('roc_auc_mean', ascending=False).iloc[0]
task_tuple = [t for t in TASKS if t[0] == best['task']][0]
_, ct, pos, neg = task_tuple
X, y, _ = build_task_dataset(counts, meta, ct, pos, neg)
model = log_reg if best['model'] == 'LogReg_L1' else rf
model.fit(X, y)
proba = model.predict_proba(X)[:,1]
thresholds = np.linspace(0.1, 0.9, 17)
curve = []
for t in thresholds:
    pred = (proba >= t).astype(int)
    curve.append({'threshold': t, 'balanced_acc': balanced_accuracy_score(y,pred), 'f1': f1_score(y,pred)})
curve = pd.DataFrame(curve)
curve.plot(x='threshold', y=['balanced_acc','f1'], marker='o', figsize=(8,5))
plt.ylim(0,1); plt.title('Threshold Sensitivity'); plt.show()

In [ ]:
pi = permutation_importance(model, X, y, n_repeats=50, random_state=RANDOM_STATE, scoring='roc_auc')
feature_mask = model.named_steps['var_filter'].get_support()
feature_names = np.array(X.columns)[feature_mask]
imp_df = pd.DataFrame({'feature': feature_names, 'importance_mean': pi.importances_mean}).sort_values('importance_mean', ascending=False)
imp_df.head(20)

## 6) Decision Framework
Convert performance + stability into action tiers:
- Tier 1: decision-ready signal
- Tier 2: promising but fragile
- Tier 3: inconclusive

## 7) Limitations and Conclusions
Document small-n constraints, uncertainty, and next-best experiments.